In [7]:
from gwcosmo import cosmology as csm
import sys
import h5py
import numpy as np
import scipy
import matplotlib.pyplot as plt
import healpy as hp
import pandas as pd
import pyarrow as pyar

In [3]:
filename = ( '/home/2619925s/'
    'samples-rpo4ab-1366933504-55469568-clipped.hdf') 

micecatpath = ('/data/wiay/galaxy_catalogs/micecat_selected_columns/'
               'micecat.parquet')

micecatdata = pd.read_parquet(micecatpath)

In [5]:
print(micecatdata)

                  ra        dec         id        z  abs_mag_r
0          26.507055  10.139941   38729316  1.26356 -20.090000
1          75.924492  54.309589  155655012  0.71115 -18.969999
2          23.831009  16.006283   19094884  1.34358 -18.959999
3          54.936234  46.859234   88989284  1.04643 -19.469999
4          86.575996  39.045132   95308388  1.02072 -19.100000
...              ...        ...        ...      ...        ...
205218232  23.979189  80.484207   17957437  1.34179 -19.110001
205218233  21.586870   8.798204   19061821  1.33565 -19.969999
205218234   8.790966  29.309452   99876157  0.99412 -19.459999
205218235  11.418935  77.039749  136908605  0.80941 -19.780001
205218236   5.276895  43.996738  152094781  0.72619 -19.730000

[205218237 rows x 5 columns]


In [ ]:
with h5py.File(filename, 'r') as obj:
    events = obj['events'][:]
    meta = dict(obj.attrs.items())

weights  = events['weights']

pdraw = 4*np.pi*np.exp(events['lnpdraw_z'] + events['lnpdraw_declination'] + events['lnpdraw_right_ascension']) #here the 4pi comes from the agnostic params of the current approach

In [15]:
# need to define an agnostic function to define z probability
# can i borrow from gwcosmo?

cosmo = csm.standard_cosmology(H0=70, Omega_m=0.3065, w0=-1., wa=0., zmax=10.0, zmin=1.e-6, zbin=5000, device="cpu", dtype=None)



In [19]:
znew = np.ascontiguousarray(events['z'], dtype=np.float64).copy()

In [26]:


pz = cosmo.p_z(znew)
pz_np = pz.detach().cpu().numpy()

In [29]:
#Need to add a far threshold

min_far = np.min([events["%s_far"%search] for search in meta["searches"]], axis=0)

detected = min_far < 0.25

#Defining the function

pdet = np.sum(weights[detected] * pz_np[detected] / pdraw[detected]) / meta['total_generated']

In [31]:
print(pdet)

0.0018158003517633268
